In [1]:
import pandas as pd
import numpy as np
import shutil

In [2]:
excel_path = r"/home/david/Git/SDLT_Number_Crunching/Data/adiya_isla_scores.xlsx"
output_path = r"/home/david/Git/SDLT_Number_Crunching/Data/consolidated_results.xlsx"

# List of sheets to evaluate (skipping overview sheets like 'Whole' if present)
sheet_mapping = {
    "CoC": "Chain of Custody",
    "Au": "Authenticity",
    "In": "Integrity",
    "Re": "Reliability",
    "La": "Lawful Acquisition",
    "Fi": "Fact vs Inference",
    "Cq": "Competence and Qualification",
    "Rr": "Repeatability and reproducibility",
    "Pc": "Privacy compliance"
}

# ---------------------------------------------------------
# 2. Configuration Settings
# ---------------------------------------------------------
annotator_cols = ["annotator1", "annotator2", "annotator3"]
scale = ["0", "1", "2"]

In [3]:
# 1. Load the Whole sheet
whole_df = pd.read_excel(excel_path, sheet_name="Whole")
whole_df["Publication"] = whole_df["Publication"].astype(str).str.strip()

print(f"{'Publication':<40} | {'Column':<35} | {'Old Value':<10} | {'New Value':<10}")
print("-" * 105)

for sheet_name, column_name in sheet_mapping.items():
    sheet = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])

    sheet.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in sheet.columns]
    )

    pub_col_tuple = [col for col in sheet.columns if col[0].lower() == "publication"][0]
    publications = sheet[pub_col_tuple].astype(str).str.strip()

    r2_annotators = [
        col for col in sheet["Round 2"].columns 
        if "annotator" in col.lower()
    ]

    round_2_data = sheet["Round 2"][r2_annotators].apply(pd.to_numeric, errors="coerce")
    medians = round_2_data.median(axis=1, skipna=True)

    median_map = pd.Series(medians.values, index=publications).dropna()

    target_col = next(
        (c for c in whole_df.columns if c.strip().lower() == column_name.lower()), 
        None
    )
    if target_col is None:
        print(f"Warning: Column '{column_name}' not found in 'Whole'. Skipping {sheet_name}.")
        continue

    new_values = whole_df["Publication"].map(median_map)
    current_numeric = pd.to_numeric(whole_df[target_col], errors="coerce")
    
    # Identify rows that actually need an update
    mismatch_mask = new_values.notna() & ~np.isclose(current_numeric, new_values, equal_nan=True)

    # Print changes before applying them
    if mismatch_mask.any():
        changed_rows = whole_df.loc[mismatch_mask, ["Publication", target_col]].copy()
        changed_rows["New_Value"] = new_values[mismatch_mask]

        for _, row in changed_rows.iterrows():
            pub = row["Publication"]
            old_val = row[target_col]
            new_val = row["New_Value"]
            print(f"{pub:<40} | {target_col:<35} | {str(old_val):<10} | {str(new_val):<10}")

        # Update whole_df
        whole_df.loc[mismatch_mask, target_col] = new_values[mismatch_mask]

# 2. Preserve all existing sheets by copying original file and replacing 'Whole'
shutil.copyfile(excel_path, output_path)

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    whole_df.to_excel(writer, sheet_name="Whole", index=False)

print("-" * 105)
print("Consolidation complete.")

Publication                              | Column                              | Old Value  | New Value 
---------------------------------------------------------------------------------------------------------
Harding_2024                             | Chain of Custody                    | 1          | 2.0       
Hao_2018                                 | Chain of Custody                    | 1          | 0.0       
Frowis_2020                              | Chain of Custody                    | 2          | 1.0       
Harding_2024                             | Authenticity                        | 1          | 2.0       
Naha_2024                                | Authenticity                        | 0          | 1.0       
Ellul_2020                               | Integrity                           | 0          | 1.0       
Akbarfam_2023                            | Reliability                         | 1          | 2.0       
Harding_2024                             | Reliability

In [4]:
whole_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   Publication                        27 non-null     str  
 1   Chain of Custody                   27 non-null     int64
 2   Authenticity                       27 non-null     int64
 3   Integrity                          27 non-null     int64
 4   Reliability                        27 non-null     int64
 5   Lawful Acquisition                 27 non-null     int64
 6   Fact vs Inference                  27 non-null     int64
 7   Competence and Qualification       27 non-null     int64
 8   Repeatability and reproducibility  27 non-null     int64
 9   Privacy compliance                 27 non-null     int64
dtypes: int64(9), str(1)
memory usage: 2.2 KB


In [ ]:
# New version


def get_row_median(row, r2_cols, r1_cols):
    """
    Extracts scores for a row:
    - Uses 'Round 2' if at least one numeric score exists; otherwise falls back to 'Round 1'.
    - Returns np.nan if neither round has valid numeric scores.
    - Resolves ties by selecting the lower score: (n - 1) // 2.
    """
    def extract_scores(cols):
        scores = []
        for col in cols:
            val = pd.to_numeric(row[col], errors="coerce")
            if pd.notna(val):
                scores.append(val)
        return scores

    active_scores = extract_scores(r2_cols)
    if not active_scores:
        active_scores = extract_scores(r1_cols)

    if not active_scores:
        return np.nan

    active_scores.sort()
    n = len(active_scores)
    return active_scores[(n - 1) // 2]


# ---------------------------------------------------------
# 1. Load Whole Sheet
# ---------------------------------------------------------
# Inspect headers to detect single vs multi-index
whole_preview = pd.read_excel(excel_path, sheet_name="Whole", nrows=2, header=None)
is_multi = False
if len(whole_preview) >= 2:
    row0 = " ".join(whole_preview.iloc[0].dropna().astype(str).str.lower())
    row1 = " ".join(whole_preview.iloc[1].dropna().astype(str).str.lower())
    if "publication" in row0 and ("author" in row1 or "category" in row1):
        is_multi = True

whole_df = pd.read_excel(excel_path, sheet_name="Whole", header=[0, 1] if is_multi else 0)

if is_multi:
    whole_df.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in whole_df.columns]
    )
    whole_author_col = next(
        (c for c in whole_df.columns if c[0].lower() == "publication" and c[1].lower() == "author"),
        next((c for c in whole_df.columns if "author" in str(c[1]).lower()), None)
    )
    whole_cat_col = next(
        (c for c in whole_df.columns if c[0].lower() == "publication" and c[1].lower() == "category"),
        None
    )
else:
    whole_author_col = next(
        (c for c in whole_df.columns if str(c).strip().lower() in ["author", "publication"]),
        whole_df.columns[0]
    )
    whole_cat_col = next((c for c in whole_df.columns if str(c).strip().lower() == "category"), None)

# Clean author strings
whole_df[whole_author_col] = whole_df[whole_author_col].astype(str).str.strip()
if whole_cat_col:
    # Forward-fill merged category cells if applicable
    whole_df[whole_cat_col] = whole_df[whole_cat_col].replace(r"^\s*$", np.nan, regex=True).ffill()

print(f"{'Publication':<40} | {'Column':<35} | {'Old Value':<10} | {'New Value':<10}")
print("-" * 105)

# Track publications that must be dropped across evaluations
publications_to_drop = set()

# ---------------------------------------------------------
# 2. Iterate Sheets & Consolidate
# ---------------------------------------------------------
for sheet_name, column_name in sheet_mapping.items():
    sheet = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])
    sheet.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in sheet.columns]
    )

    # Locate Publication -> Author column
    sheet_author_col = next(
        (c for c in sheet.columns if c[0].lower() == "publication" and c[1].lower() == "author"),
        next((c for c in sheet.columns if c[0].lower() == "publication" and "category" not in c[1].lower()), None)
    )
    if not sheet_author_col:
        print(f"Warning: Could not identify Author column in '{sheet_name}'. Skipping.")
        continue

    # Identify Annotator columns
    r2_annotators = [c for c in sheet.columns if c[0].lower() == "round 2" and "annotator" in c[1].lower()]
    r1_annotators = [c for c in sheet.columns if c[0].lower() == "round 1" and "annotator" in c[1].lower()]

    median_map = {}
    for _, row in sheet.iterrows():
        pub = str(row[sheet_author_col]).strip()
        if not pub or pub.lower() in ["nan", "none"]:
            continue

        score = get_row_median(row, r2_annotators, r1_annotators)
        if pd.isna(score):
            # Row has no inputs in either round -> register for removal
            publications_to_drop.add(pub)
        else:
            median_map[pub] = score

    # Find matching target column in whole_df safely
    if is_multi:
        target_col = next(
            (c for c in whole_df.columns if str(c[0]).strip().lower() == column_name.strip().lower() or 
                                           str(c[1]).strip().lower() == column_name.strip().lower()),
            None
        )
    else:
        target_col = next(
            (c for c in whole_df.columns if str(c).strip().lower() == column_name.strip().lower()),
            None
        )

    if target_col is None:
        print(f"Warning: Column '{column_name}' not found in 'Whole'. Skipping {sheet_name}.")
        continue

    new_values = whole_df[whole_author_col].map(median_map)
    current_numeric = pd.to_numeric(whole_df[target_col], errors="coerce")

    # Update mismatched values
    mismatch_mask = new_values.notna() & ~np.isclose(current_numeric, new_values, equal_nan=True)

    if mismatch_mask.any():
        changed_rows = whole_df.loc[mismatch_mask, [whole_author_col, target_col]].copy()
        changed_rows["New_Value"] = new_values[mismatch_mask]

        col_display = target_col if isinstance(target_col, str) else " - ".join(target_col)
        for _, row in changed_rows.iterrows():
            pub = row[whole_author_col]
            old_val = row[target_col]
            new_val = row["New_Value"]
            print(f"{pub:<40} | {col_display:<35} | {str(old_val):<10} | {str(new_val):<10}")

        whole_df.loc[mismatch_mask, target_col] = new_values[mismatch_mask]

# ---------------------------------------------------------
# 3. Drop Unannotated Rows from Final Dataframe
# ---------------------------------------------------------
if publications_to_drop:
    drop_mask = whole_df[whole_author_col].isin(publications_to_drop)
    if drop_mask.any():
        dropped_count = drop_mask.sum()
        whole_df = whole_df.loc[~drop_mask].reset_index(drop=True)
        print(f"\nDropped {dropped_count} row(s) with no inputs in Round 1 or Round 2: {publications_to_drop}")

# ---------------------------------------------------------
# 4. Save Finalized Dataframe
# ---------------------------------------------------------
shutil.copyfile(excel_path, output_path)

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    whole_df.to_excel(writer, sheet_name="Whole", index=False)

print("-" * 105)
print("Consolidation complete.")